# Part 1 : Implement a CNN model from scratch

we are training the model to Classify aircraft images into aircraft **variants**. (e.g., Boeing 737-800, Airbus A320-214)

In [2]:
import time
import torch
import torchvision
import numpy as np
# from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

In [7]:
BATCH_SIZE = 32
NUM_CLASSES = 100
EPOCHS = 40
LR = 1e-2
IMAGE_SIZE = 128

In [10]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Load the data :

In [8]:
# transformation pipeline :
# ===== CELL 3: Data Augmentation =====
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

# load dataset
train_dataset = torchvision.datasets.FGVCAircraft(
    root="data/fgvc_aircraft",   # where data will be downloaded/stored
    split="train",
    transform=train_transform,
    download=True
)

val_dataset = torchvision.datasets.FGVCAircraft(
    root="data/fgvc_aircraft",
    split="val",
    transform=val_transform,
    download=False
)

test_dataset = torchvision.datasets.FGVCAircraft(
    root="data/fgvc_aircraft",
    split="test",
    transform=val_transform,
    download=False
)

# data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [9]:
print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")
print(f"Validation size: {len(val_dataset)}")

Train size: 3334
Test size: 3333
Validation size: 3333


## Define the model architecture :

In [11]:

# ===== CELL 5: Model Definition =====
class ScratchNet100(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x

model = ScratchNet100(NUM_CLASSES).to(DEVICE)

# ===== CELL 6: Loss & Optimizer =====
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# ===== CELL 7: Training & Validation Functions =====
def train_one_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        preds = model(x)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (preds.argmax(1) == y).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def validate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            preds = model(x)
            correct += (preds.argmax(1) == y).sum().item()
    return correct / len(loader.dataset)

# ===== CELL 8: Training Loop =====
for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_acc = validate(model, val_loader)
    scheduler.step()

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Loss: {train_loss:.4f} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Val Acc: {val_acc:.4f}")


Epoch 1/40 | Loss: 4.7466 | Train Acc: 0.0150 | Val Acc: 0.0240
Epoch 2/40 | Loss: 4.5238 | Train Acc: 0.0261 | Val Acc: 0.0228
Epoch 3/40 | Loss: 4.4807 | Train Acc: 0.0294 | Val Acc: 0.0279
Epoch 4/40 | Loss: 4.4570 | Train Acc: 0.0261 | Val Acc: 0.0333
Epoch 5/40 | Loss: 4.4253 | Train Acc: 0.0345 | Val Acc: 0.0303
Epoch 6/40 | Loss: 4.4218 | Train Acc: 0.0318 | Val Acc: 0.0282
Epoch 7/40 | Loss: 4.4105 | Train Acc: 0.0375 | Val Acc: 0.0369
Epoch 8/40 | Loss: 4.3930 | Train Acc: 0.0420 | Val Acc: 0.0288
Epoch 9/40 | Loss: 4.3782 | Train Acc: 0.0393 | Val Acc: 0.0387
Epoch 10/40 | Loss: 4.3553 | Train Acc: 0.0399 | Val Acc: 0.0429
Epoch 11/40 | Loss: 4.3002 | Train Acc: 0.0498 | Val Acc: 0.0366
Epoch 12/40 | Loss: 4.2753 | Train Acc: 0.0498 | Val Acc: 0.0498
Epoch 13/40 | Loss: 4.2601 | Train Acc: 0.0513 | Val Acc: 0.0492
Epoch 14/40 | Loss: 4.2285 | Train Acc: 0.0528 | Val Acc: 0.0522
Epoch 15/40 | Loss: 4.1888 | Train Acc: 0.0606 | Val Acc: 0.0501
Epoch 16/40 | Loss: 4.1641 | Train

## Evaluet the model on validation data :

In [ ]:
test_loss, test_accuracy = evaluate_model(
    model,
    test_loader,
    criterion,
    device,
    name="Test data"
)

print(f"Test loss : {test_loss}")
print(f"Test accuracy : {test_accuracy}")


Test loss : 5.376230136553446
Test accuracy : 12.901290129012901


___